In [1]:
import pandas as pd
import numpy as np

In [ ]:
import glob
collection = glob.glob("**/orders_2025_*.csv", recursive=True)

# Defining a function for data cleaning

def cleaning_data(path):
    df = pd.read_csv(path, na_values = ["TBD","unknown","-"], dtype=str)
    df_copy  = df.copy()
    df_copy["Customer ID"] = df_copy["Customer ID"].str.strip()
    df_copy["SKU"] = df_copy["SKU"].str.strip()
    df_copy["SKU"] = df_copy["SKU"].str.title()
    df_copy["Product Name"] = df_copy["Product Name"].str.strip()
    df_copy["Product Name"] = df_copy["Product Name"].str.title()
    df_copy["Category"] = df_copy["Category"].str.strip()
    df_copy["Category"] = df_copy["Category"].str.title()
    df_copy["Quantity"] = df_copy["Quantity"].str.strip()
    df_copy["Quantity"] = df_copy["Quantity"].str.replace("-1","1", regex = True)
    df_copy["Quantity"] = df_copy["Quantity"].str.replace("two","2", regex = True)
    df_copy["Quantity"] = pd.to_numeric(df_copy["Quantity"])
    df_copy["Unit Price"] = df_copy["Unit Price"].str.replace("Rs ","", regex = True)
    df_copy["Unit Price"] = df_copy["Unit Price"].str.replace(",","", regex = True)
    df_copy["Unit Price"] = pd.to_numeric(df_copy["Unit Price"])
    df_copy["Discount %"] = df_copy["Discount %"].str.replace("10%","10", regex = True)
    df_copy["Order Status"] = df_copy["Order Status"].str.title()
    df_copy["Order Status"] = df_copy["Order Status"].str.strip()
    df_copy["Sales Channel"] = df_copy["Sales Channel"].str.title()
    df_copy["Sales Channel"] = df_copy["Sales Channel"].str.strip()
    df_copy["Discount %"] = pd.to_numeric(df_copy["Discount %"])
    df_copy["first"] = pd.to_datetime(df_copy["Order Date"], format = "%d/%m/%Y", errors = "coerce")
    df_copy["second"] = pd.to_datetime(df_copy["Order Date"], format = "%Y-%m-%d", errors = "coerce")
    df_copy["third"] = pd.to_datetime(df_copy["Order Date"], format = "%d-%m-%Y", errors = "coerce")
    df_copy['order_date'] = df_copy['first'].combine_first(df_copy['second']).combine_first(df_copy['third'])
    df_copy = df_copy.drop(columns = ["Order Date","first","second","third"])
    df_copy = df_copy.rename(columns = {"Order ID":"order_id","Customer ID":"customer_id","SKU":"sku","Product Name":"product_name","Category":"category","Quantity":"quantity","Unit Price":"unit_price","Discount %":"discount%","Order Status":"order_status","Sales Channel":"sales_channel","Order City":"order_city","Payment Method":"payment_mathod"})
    #df_copy.columns = df_copy.columns.str.lower()
    return df_copy

# Cleaning and appending

collection
cleaned_data = []
rough_data = []
row_count = 0
for each in collection:
    rough =  pd.read_csv(each)
    rough_data.append(rough)
    row_count = row_count + pd.read_csv(each).shape[0]
    select = cleaning_data(each)
    select["path"] = each
    cleaned_data.append(select)
cleaned_data
cross_ref = pd.concat(rough_data, ignore_index = True)
final_format = pd.concat(cleaned_data, ignore_index = True)
final_format["path"] = final_format["path"].str.extract(r'\w+_(\d+_\d+).csv')
final_format

assert final_format.shape[0] == row_count, "Missing Data"


#cleaning and Merging customers to Final_format

customers_path = glob.glob("**/customers.csv",recursive = True)

customers = pd.read_csv(customers_path[0])
customers["Customer ID"] = customers["Customer ID"].str.upper()
customers["Customer ID"] = customers["Customer ID"].str.strip()
customers["Customer Name"] = customers["Customer Name"].str.title()
customers["Customer Name"] = customers["Customer Name"].str.strip()
customers["Signup Date"] = pd.to_datetime(customers["Signup Date"], format = "%d/%m/%Y" ) #, format = "%d-%m-Y%" format = "%Y-%m-%d"
customers = customers.drop_duplicates()
yo = customers.value_counts("Customer ID").reset_index()
yoyo = sorted(yo[yo["count"] > 1]["Customer ID"].unique())
customers[customers["Customer ID"].isin(yoyo)].sort_values("Customer ID")

# Managing and Filling less important values

big = final_format.merge(customers, left_on = "customer_id" , right_on = "Customer ID", how = "left")
big = big.drop(columns = ["Customer ID"])
big.columns = big.columns.str.lower()
big["customer_id"] = big["customer_id"].fillna("unknown")
count_of_not_matched = big[(big["customer_id"] != "unknown") & (big["customer name"].isnull())]
big["discount%"] = big["discount%"].fillna(0)

# Handling important missing values

null_prices = big[big["unit_price"].isnull()]
completed = big[big["order_status"] == "Completed"]
completed[(completed["product_name"] == "Desk Lamp") & (completed["unit_price"].isnull())]

# Testing

test_set = completed.groupby("product_name").agg(Total_count = ("product_name","count"),
                                      Total_missing_values = ("product_name",lambda x : x[completed.loc[x.index , "unit_price"].isnull()].count())).reset_index().sort_values("Total_count", ascending = False)
test_set["%Missing_data"] = (test_set["Total_missing_values"]/ test_set["Total_count"]) * 100
test_set.sort_values("%Missing_data", ascending = False)

# Application and Unit Price is resolved

completed["median"] = completed.groupby("product_name")["unit_price"].transform("median")
completed["unit_price"] = completed["unit_price"].fillna(completed["median"])
completed

# Coming to quantity issues and missing data

cc = completed[completed["quantity"].isnull()]
completed["mean"] = completed.groupby("product_name")["quantity"].transform("mean")
completed["quantity"] = completed["quantity"].fillna(completed["mean"])


# Moving on to revenue part and related analysis

completed = completed.drop(columns = ["median","mean"])
completed["revenue"] = completed["unit_price"] * completed["quantity"] * (1 - (completed["discount%"]/100))
completed["month"] = completed["order_date"].dt.month
best_performing_month = completed.groupby("month")["revenue"].sum().reset_index().sort_values("revenue", ascending = False).head(1)
best_performing_sku = completed.groupby("sku")["revenue"].sum().reset_index().sort_values("revenue", ascending = False).head(1)

sales_channel_revenue = round(completed.groupby("sales_channel")["revenue"].sum().reset_index().sort_values("revenue", ascending = False),2)
customer_segment_revnue = round(completed.groupby("customer segment")["revenue"].sum().reset_index().sort_values("revenue", ascending = False),2)
region_revnue = round(completed.groupby("region")["revenue"].sum().reset_index().sort_values("revenue", ascending = False),2)

sales_channel_revenue["Total"] = sales_channel_revenue["revenue"].sum()
customer_segment_revnue["Total"] = customer_segment_revnue["revenue"].sum()
region_revnue["Total"] = region_revnue["revenue"].sum()

sales_channel_revenue["Contribution"] = round(sales_channel_revenue["revenue"] / sales_channel_revenue["Total"],2) * 100
customer_segment_revnue["Contribution"] = round(customer_segment_revnue["revenue"] / customer_segment_revnue["Total"],2)* 100
region_revnue["Contribution"] = round(region_revnue["revenue"] / region_revnue["Total"],2)* 100

#Weekday analysis

completed["weekday"] = completed["order_date"].dt.day_name()
Day_sales = round(completed.groupby("weekday")["revenue"].sum().reset_index().sort_values("revenue", ascending = False),2)
Day_sales



,weekday,revenue
3,Sunday,1266032.78
0,Friday,1262887.03
4,Thursday,1217261.40
6,Wednesday,1209126.51
2,Saturday,1113983.21
5,Tuesday,1083865.42
1,Monday,1057364.14
